# Phase 1 — Stratified Subset Generation
This notebook creates **3–5 manageable subsets** (~50k–100k docs) by:
- Clustering queries via MiniLM embeddings + KMeans (proportional sampling)
- Preserving document-length diversity via binning

> Matches SOP Step 2 (Create Stratified Subsets).

In [1]:
import sys, subprocess, importlib.util

# 1) Remove the broken installs in THIS kernel
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "sentence-transformers", "transformers", "tokenizers", "huggingface-hub"], check=False)

# 2) Reinstall compatible, stable versions (no cache to avoid partial wheels)
env = {"PIP_NO_CACHE_DIR": "1"}
subprocess.run([sys.executable, "-m", "pip", "install",
                "transformers==4.44.2",           # solid 4.x release
                "sentence-transformers==3.0.1",
                "huggingface-hub>=0.24.0",
                "tokenizers>=0.15,<0.23",         # keeps ABI compatible with 4.44.x
                "safetensors>=0.4.3"], env=env, check=True)

# 3) Sanity check the exact symbols that failed before
code = "from transformers import PreTrainedModel, TrainingArguments; print('OK transformers'); " \
       "from sentence_transformers import SentenceTransformer; print('OK sentence-transformers')"
subprocess.run([sys.executable, "-c", code], check=True)

print("Reinstall done.")


Found existing installation: sentence-transformers 3.0.1
Uninstalling sentence-transformers-3.0.1:
  Successfully uninstalled sentence-transformers-3.0.1
Found existing installation: transformers 4.44.2
Uninstalling transformers-4.44.2:
  Successfully uninstalled transformers-4.44.2
Found existing installation: tokenizers 0.19.1
Uninstalling tokenizers-0.19.1:
  Successfully uninstalled tokenizers-0.19.1
Found existing installation: huggingface-hub 0.36.0
Uninstalling huggingface-hub-0.36.0:
  Successfully uninstalled huggingface-hub-0.36.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 7.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 8.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 4.7 MB/s  0:00:00



/home/mallarapuhemavarshini/anaconda3/lib/python3.12/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
There was a problem when trying to write in your cache folder (/path/with/space/hf-cache/transformers). You should set the environment variable TRANSFORMERS_CACHE to a writable directory.


OK transformers
OK sentence-transformers
Reinstall done.


In [2]:
# Optional: install
!pip install -r requirements.txt

In [3]:
import os
from pathlib import Path

# define writable cache directories inside your home directory
cache_base = Path.home() / ".cache" / "huggingface"
os.environ["HF_HOME"] = str(cache_base)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(cache_base / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(cache_base / "transformers")
os.environ["SENTENCE_TRANSFORMERS_HOME"] = str(cache_base / "sentence-transformers")

# make sure they exist
for p in [
    cache_base,
    cache_base / "hub",
    cache_base / "transformers",
    cache_base / "sentence-transformers",
]:
    p.mkdir(parents=True, exist_ok=True)

In [4]:
# ---------- CHANGES START HERE ----------
from __future__ import annotations
import json, random
from pathlib import Path
from typing import Dict, List, Set
import numpy as np
from tqdm import tqdm
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
# Use a single seed everywhere
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
WORK_DIR = Path("./work")
DATASET = "beir/trec-covid"
NUM_SUBSETS = 3
QUERY_CLUSTERS = 12
TARGET_DOCS_PER_SUBSET = 56000
ds_dir = WORK_DIR / "datasets" / DATASET.replace("/", "_")
subs_root = WORK_DIR / "subsets" / DATASET.replace("/", "_")
subs_root.mkdir(parents=True, exist_ok=True)

def read_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

corpus_path = ds_dir / "corpus.jsonl"
queries_path = ds_dir / "queries.jsonl"
qrels_path = ds_dir / "qrels.jsonl"

# Load into memory structures
corpus_map = {r["doc_id"]: r for r in read_jsonl(corpus_path)}
qmap = {r["qid"]: r["text"] for r in read_jsonl(queries_path)}
qrels = {}
for r in read_jsonl(qrels_path):
    qrels.setdefault(r["qid"], {})[r["doc_id"]] = int(r["rel"])
# 1) Compute per-subset query targets to cover ALL queries (e.g., 50 -> [17,17,16])
all_qids = list(qmap.keys())
Q = len(all_qids)
K = NUM_SUBSETS
q_sizes = [Q // K + (1 if i < (Q % K) else 0) for i in range(K)]  # e.g., [17,17,16]

# 2) Stratify by query clusters to keep a balanced mix per subset
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
qids_shuf = all_qids[:]
random.shuffle(qids_shuf)  # stable randomness
qtexts = [qmap[qid] for qid in qids_shuf]
qemb = model.encode(qtexts, batch_size=256, show_progress_bar=False,
                    convert_to_numpy=True, normalize_embeddings=True)

kmeans = KMeans(n_clusters=QUERY_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
labels = kmeans.fit_predict(qemb)

# Build cluster -> [qids] map (using shuffled order)
cluster_bins: Dict[int, List[str]] = {}
for qid, lab in zip(qids_shuf, labels):
    cluster_bins.setdefault(int(lab), []).append(qid)

# 3) Round-robin fill subsets from each cluster up to q_sizes
qids_by_subset: List[List[str]] = [[] for _ in range(K)]
need = q_sizes[:]  # remaining capacity per subset

for lab, qlist in cluster_bins.items():
    idx = 0
    for qid in qlist:
        # find next subset that still needs queries
        tries = 0
        while need[idx] == 0 and tries < K:
            idx = (idx + 1) % K
            tries += 1
        if need[idx] == 0:
            continue  # this cluster still has qids but all subsets are full
        qids_by_subset[idx].append(qid)
        need[idx] -= 1
        idx = (idx + 1) % K

# Sanity: ensure exact sizes
assert all(len(qids_by_subset[i]) == q_sizes[i] for i in range(K)), (q_sizes, [len(x) for x in qids_by_subset])

# 4) Build and write each subset
used_docs: Set[str] = set()

def write_subset(sub_dir: Path, sampled_qids: List[str]):
    sub_dir.mkdir(parents=True, exist_ok=True)

    # Seed doc pool with positives for these queries
    doc_pool: Set[str] = set()
    for qid in sampled_qids:
        for did, rel in qrels.get(qid, {}).items():
            if rel > 0:
                doc_pool.add(did)

    # Top up to TARGET_DOCS_PER_SUBSET with unused docs (no overlaps across subsets)
    if len(doc_pool) < TARGET_DOCS_PER_SUBSET:
        need = TARGET_DOCS_PER_SUBSET - len(doc_pool)
        # prefer unused docs; if exhausted, allow remaining
        candidates = [d for d in corpus_map.keys() if d not in used_docs and d not in doc_pool]
        random.shuffle(candidates)
        take = candidates[:max(0, need)]
        doc_pool.update(take)

        # fallback if still short (rare)
        if len(doc_pool) < TARGET_DOCS_PER_SUBSET:
            more_need = TARGET_DOCS_PER_SUBSET - len(doc_pool)
            remaining = [d for d in corpus_map.keys() if d not in doc_pool]
            random.shuffle(remaining)
            doc_pool.update(remaining[:more_need])

    # Mark used to reduce overlap across subsets
    used_docs.update(doc_pool)

    # Write queries.jsonl (only this subset’s queries)
    with (sub_dir / "queries.jsonl").open("w", encoding="utf-8") as f:
        for qid in sampled_qids:
            f.write(json.dumps({"qid": str(qid), "text": qmap[str(qid)]}, ensure_ascii=False) + "\n")

    # Write corpus.jsonl (only this subset’s docs)
    with (sub_dir / "corpus.jsonl").open("w", encoding="utf-8") as f:
        for did in doc_pool:
            if did in corpus_map:
                f.write(json.dumps(corpus_map[did], ensure_ascii=False) + "\n")

    # Write qrels.jsonl filtered to (subset queries x subset docs)
    doc_pool_set = set(doc_pool)
    with (sub_dir / "qrels.jsonl").open("w", encoding="utf-8") as f:
        for qid in sampled_qids:
            for did, rel in qrels.get(qid, {}).items():
                if did in doc_pool_set and rel > 0:
                    f.write(json.dumps({"qid": str(qid), "doc_id": str(did), "rel": int(rel)}, ensure_ascii=False) + "\n")

    # Quick checks
    qs = {json.loads(x)["qid"] for x in (sub_dir / "queries.jsonl").open("r", encoding="utf-8")}
    assert len(qs) == len(sampled_qids), f"query count mismatch in {sub_dir}"
    print(f"[{sub_dir.name}] queries={len(sampled_qids)} docs={len(doc_pool)}")

# 5) Write all subsets (expect sizes like [17, 17, 16])
for si in range(K):
    sub_dir = subs_root / f"subset_{si+1}"
    write_subset(sub_dir, qids_by_subset[si])

# ---------- CHANGES END HERE ----------


/home/mallarapuhemavarshini/anaconda3/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/home/mallarapuhemavarshini/anaconda3/lib/python3.12/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/home/mallarapuhemavarshini/anaconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


[subset_1] queries=17 docs=56000
[subset_2] queries=17 docs=56000
[subset_3] queries=16 docs=56000


In [5]:
print("Dataset directory:", ds_dir)
print("Queries file exists:", (ds_dir / 'queries.jsonl').exists())


Dataset directory: work/datasets/beir_trec-covid
Queries file exists: True


In [6]:
qmap = {r["qid"]: r["text"] for r in read_jsonl(queries_path)}
print("Total queries loaded:", len(qmap))
print("Sample:", list(qmap.items())[:3])


Total queries loaded: 50
Sample: [('1', 'what is the origin of covid-19'), ('2', 'how does the coronavirus respond to changes in the weather'), ('3', 'will sars-cov2 infected people develop immunity? is cross protection possible?')]


In [36]:
import torch, os, subprocess, sys
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    # optional: basic memory info
    try:
        print(subprocess.check_output(["nvidia-smi", "--query-gpu=memory.total,memory.used", "--format=csv,noheader"]).decode())
    except Exception as e:
        print("nvidia-smi not available:", e)
else:
    print("No CUDA visible to PyTorch.")


torch: 2.4.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
4096 MiB, 2667 MiB



In [7]:
# Build query embeddings + KMeans clusters
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)
qids = list(qmap.keys())
qtexts = [qmap[qid] for qid in qids]
#qemb = model.encode(qtexts, batch_size=256, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
qemb = model.encode(qtexts, batch_size=256, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)

kmeans = KMeans(n_clusters=QUERY_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
labels = kmeans.fit_predict(qemb)

buckets = {}
for qid, lab in zip(qids, labels):
    buckets.setdefault(int(lab), []).append(qid)

/home/mallarapuhemavarshini/anaconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:
# Doc-length bins for diversity
import numpy as np
lengths, docids = [], []
for d in corpus_map.values():
    lengths.append(int(d["len"])); docids.append(d["doc_id"])
lengths = np.array(lengths)
edges = np.quantile(lengths, q=np.linspace(0, 1, 6))
bins = {i: [] for i in range(5)}
for did, L in zip(docids, lengths):
    idx = int(np.searchsorted(edges, L, side="right") - 1)
    idx = min(max(idx, 0), 4)
    bins[idx].append(did)

In [10]:
import numpy as np, random, json
from pathlib import Path

random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

# --- load maps (keep your existing read_jsonl) ---
# corpus_map: {doc_id: {...}}
# qmap: {qid: text}
# qrels: {qid: {doc_id: rel, ...}}

all_qids = list(qmap.keys())
random.shuffle(all_qids)

# Split queries into disjoint chunks (balanced)
q_chunks = np.array_split(all_qids, NUM_SUBSETS)

used_docs = set()
subs_root = Path("./work/subsets/beir_trec-covid")
subs_root.mkdir(parents=True, exist_ok=True)

for si in range(NUM_SUBSETS):
    sub_dir = subs_root / f"subset_{si+1}"
    sub_dir.mkdir(parents=True, exist_ok=True)

    # Pick queries for this subset (cap per target)
    qids_chunk = list(q_chunks[si])
    sampled_qids = list(q_chunks[si])  # keep all queries assigned to that chunk

    #sampled_qids = qids_chunk[:min(TARGET_QUERIES_PER_SUBSET, len(qids_chunk))]

    # Build doc pool: positives from qrels for these queries
    doc_pool = set()
    for qid in sampled_qids:
        for did, rel in qrels.get(qid, {}).items():
            if rel > 0:
                doc_pool.add(did)

    # Top up with unused docs to reach TARGET_DOCS_PER_SUBSET
    if len(doc_pool) < TARGET_DOCS_PER_SUBSET:
        need = TARGET_DOCS_PER_SUBSET - len(doc_pool)
        candidates = [d for d in corpus_map.keys() if d not in used_docs and d not in doc_pool]
        random.shuffle(candidates)
        doc_pool.update(candidates[:max(0, need)])

    # Update global used set to reduce overlap across subsets
    used_docs.update(doc_pool)

    # Write files
    with (sub_dir / "queries.jsonl").open("w", encoding="utf-8") as f:
        for qid in sampled_qids:
            f.write(json.dumps({"qid": qid, "text": qmap[qid]}, ensure_ascii=False) + "\n")

    with (sub_dir / "corpus.jsonl").open("w", encoding="utf-8") as f:
        for did in doc_pool:
            if did in corpus_map:
                f.write(json.dumps(corpus_map[did], ensure_ascii=False) + "\n")

    doc_pool_set = set(doc_pool)
    with (sub_dir / "qrels.jsonl").open("w", encoding="utf-8") as f:
        for qid in sampled_qids:
            for did, rel in qrels.get(qid, {}).items():
                if did in doc_pool_set and rel > 0:
                    f.write(json.dumps({"qid": qid, "doc_id": did, "rel": int(rel)}, ensure_ascii=False) + "\n")

    print(f"[subset {si+1}] queries={len(sampled_qids)} docs={len(doc_pool)} -> {sub_dir}")


[subset 1] queries=17 docs=56000 -> work/subsets/beir_trec-covid/subset_1
[subset 2] queries=17 docs=56000 -> work/subsets/beir_trec-covid/subset_2
[subset 3] queries=16 docs=56000 -> work/subsets/beir_trec-covid/subset_3


In [11]:
print("Loading from:", corpus_path.resolve())
num_lines = sum(1 for _ in open(corpus_path, "rb"))
print("Actual lines in corpus.jsonl:", num_lines)
print("len(corpus_map):", len(corpus_map))

Loading from: /home/mallarapuhemavarshini/Desktop/Mystudies/ir/git/information_retrieval/work/datasets/beir_trec-covid/corpus.jsonl
Actual lines in corpus.jsonl: 170000
len(corpus_map): 170000
